# Validated curation pipeline demo

This notebook demonstrates the validation-first pipeline. The LLM is treated only as a proposal engine; deterministic validation and review determine acceptance.

In [ ]:
from pathlib import Path
from fair_metadata_curation.curation.pipeline import CurationPipeline
from fair_metadata_curation.io import load_jsonl
from fair_metadata_curation.llm.mock_provider import MockProvider
from fair_metadata_curation.ontology.local_cache import LocalCacheResolver
from fair_metadata_curation.export.json_export import export_proposals
from fair_metadata_curation.export.jsonld_export import export_jsonld
from fair_metadata_curation.export.report import generate_markdown_report


In [ ]:
repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
pipeline = CurationPipeline(
    llm=MockProvider(repo_root / 'tests' / 'fixtures' / 'mock_llm_response.json'),
    resolver=LocalCacheResolver(repo_root / 'tests' / 'fixtures' / 'ontology_cache.json'),
    config={'schema_path': repo_root / 'src' / 'fair_metadata_curation' / 'schemas' / 'biosample_cedar.schema.json'}
)
records = load_jsonl(repo_root / 'examples' / 'input_records.jsonl')
proposals = pipeline.run_batch(records)
[(p.record_id, p.validation_report.passed, p.validation_report.warnings) for p in proposals]


In [ ]:
export_proposals(proposals, repo_root / 'examples' / 'curated_output.jsonl')
export_jsonld(proposals, repo_root / 'examples' / 'curated_output.jsonld')
Path(repo_root / 'examples' / 'validation_report.md').write_text(generate_markdown_report(proposals), encoding='utf-8')
